In [0]:
import requests
import pandas as pd 
import pyspark as ps
import time
from pyspark.sql.functions import col, sum as spark_sum, isnan, when, count

In [0]:
indicators = ['NCDMORT3070', 'SDGPM25', 'WHOSIS_000001', 'WHOSIS_000002', 'MDG_0000000029', 'HIV_0000000026', 'NCD_GLUC_04', 'BP_04', 'NCD_BMI_30C', 'M_Est_smk_curr_std', 'WSH_WATER_SAFELY_MANAGED', 'WSH_SANITATION_SAFELY_MANAGED', 'NUTRITION_ANT_HAZ_NE2', 'NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV', 'GHED_GGHE-D_pc_US_SHA2011']

In [0]:
import time
import requests
import pandas as pd

base_url = "https://ghoapi.azureedge.net/api/"
dfs = []
erros = []

for code in indicators:
    url = f"{base_url}{code}"
    try:
        response = requests.get(url, timeout=30)
        data = response.json()
        df_temp = pd.DataFrame(data['value'])
        df_temp['IndicatorCode'] = code
        dfs.append(df_temp)
        print(f"{code} — {len(df_temp)} linhas")
    except Exception as e:
        print(f"{code} — erro: {e}")
        erros.append(code)
    
    time.sleep(2)

df_pandas = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {df_pandas.shape}")
print(f"Erros: {erros}")

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

df_spark = spark.createDataFrame(df_pandas)

df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.gho_raw")

print(f"Salvo em workspace.bronze.gho_raw — {df_spark.count()} linhas")

In [0]:
df_check = spark.read.table("workspace.bronze.gho_raw")

print(f"Total de linhas: {df_check.count()}")

df_check.groupBy("IndicatorCode").count().orderBy("IndicatorCode").show(20, truncate=False)